# Airline Delay Visualization Analysis — 12 charts

Notebook này được tạo để chạy trên **Google Colab** và dùng làm minh chứng nhóm tự vẽ biểu đồ cho project cuối kì môn **Biểu diễn trực quan dữ liệu**.

Dataset sử dụng: `Airline_Delay_Cause_clean.csv`  
Kỳ dữ liệu: **Tháng 04/2026**  
Câu hỏi trung tâm: **Nếu hành khách thường xuyên đi máy bay muốn giảm rủi ro delay, nên chú ý hãng bay, sân bay và nhóm nguyên nhân nào?**

Notebook có 12 chart, không chỉ lặp lại một loại biểu đồ. Các chart được chia theo nhóm mục tiêu trực quan hóa:

| Nhóm mục tiêu | Chart trong notebook |
|---|---|
| Scope / KPI baseline | Chart 01 |
| Distribution understanding | Chart 02, 03, 04 |
| Comparison | Chart 05, 06 |
| Relationship / decision support | Chart 07, 08 |
| Composition / part-to-whole | Chart 09, 10, 11 |
| Priority matrix | Chart 12 |

> Ghi chú lý thuyết: Dataset hiện tại chỉ có **một tháng** nên notebook **không gượng ép line chart/time-series trend**. Theo lý thuyết biểu đồ xu hướng, line chart phù hợp khi trục X là thời gian liên tục và có nhiều điểm thời gian. Nếu nhóm tải thêm nhiều tháng từ BTS, có thể bổ sung chart trend sau.

## 0. Cài thư viện và load dữ liệu

Cell này dùng được cả trên Colab lẫn khi chạy local. Nếu chạy trên Colab, hãy upload file `Airline_Delay_Cause_clean.csv` khi được hỏi.

In [ ]:
# Nếu chạy trên Google Colab, cell này sẽ tự yêu cầu upload file nếu chưa thấy CSV trong /content.
import os
import glob
import math
import zipfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)

# Tạo folder lưu minh chứng chart dạng html
OUTPUT_DIR = 'airline_delay_chart_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hàm lưu biểu đồ thành HTML để nộp minh chứng nếu cần
saved_charts = []
def save_fig(fig, filename):
    path = os.path.join(OUTPUT_DIR, f'{filename}.html')
    fig.write_html(path, include_plotlyjs='cdn')
    saved_charts.append(path)
    return path

# Load file dữ liệu
candidate_paths = [
    '/content/Airline_Delay_Cause_clean.csv',
    '/content/Airline_Delay_Cause_clean - Airline_Delay_Cause_clean.csv.csv',
    '/mnt/data/Airline_Delay_Cause_clean - Airline_Delay_Cause_clean.csv.csv',
    '/mnt/data/airline_delay_cleaned/Airline_Delay_Cause_clean.csv',
    'Airline_Delay_Cause_clean.csv',
    'Airline_Delay_Cause_clean - Airline_Delay_Cause_clean.csv.csv'
]
# Fallback: tự tìm mọi file CSV có tên chứa Airline_Delay_Cause_clean
candidate_paths += glob.glob('/content/*Airline_Delay_Cause_clean*.csv')
candidate_paths += glob.glob('/mnt/data/*Airline_Delay_Cause_clean*.csv')
candidate_paths += glob.glob('*Airline_Delay_Cause_clean*.csv')
# Loại trùng nhưng vẫn giữ thứ tự ưu tiên
candidate_paths = list(dict.fromkeys(candidate_paths))

csv_path = None
for p in candidate_paths:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    try:
        from google.colab import files
        print('Chưa thấy file Airline_Delay_Cause_clean.csv. Hãy upload file CSV đã làm sạch.')
        uploaded = files.upload()
        csv_candidates = [name for name in uploaded.keys() if name.endswith('.csv')]
        if not csv_candidates:
            raise FileNotFoundError('Không có file .csv nào được upload.')
        csv_path = csv_candidates[0]
    except Exception as e:
        raise FileNotFoundError('Không tìm thấy Airline_Delay_Cause_clean.csv. Hãy upload file trước khi chạy notebook.') from e

df = pd.read_csv(csv_path)
print('Loaded:', csv_path)
print('Shape:', df.shape)
df.head()

Chưa thấy file Airline_Delay_Cause_clean.csv. Hãy upload file CSV đã làm sạch.


Saving Airline_Delay_Cause_clean - Airline_Delay_Cause_clean.csv.csv to Airline_Delay_Cause_clean - Airline_Delay_Cause_clean.csv.csv
Loaded: Airline_Delay_Cause_clean - Airline_Delay_Cause_clean.csv.csv
Shape: (1857, 42)


,carrier,carrier_name,airport,airport_name,airport_city,airport_state,airport_full_name,arr_flights,arr_del15,arr_cancelled,arr_diverted,arr_delay,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,delay_rate,cancel_rate,diverted_rate,problem_rate,avg_delay_min_per_delayed_flight,avg_delay_min_per_arrival,carrier_ct_share,weather_ct_share,nas_ct_share,security_ct_share,late_aircraft_ct_share,carrier_delay_share,weather_delay_share,nas_delay_share,security_delay_share,late_aircraft_delay_share,cause_count_sum,cause_delay_min_sum,cause_count_diff,cause_delay_min_diff
0,G4,Allegiant Air,MCO,"Orlando, FL: Orlando International",Orlando,FL,Orlando International,37.0000,4.0000,1.0000,0.0000,238.0000,1.0000,1.0000,1.1000,0.0000,0.9000,38.0000,107.0000,49.0000,0.0000,44.0000,10.810.810.810.810.800,2.702.702.702.702.700,0.0,13.513.513.513.513.500,59.5,64.324.324.324.324.300,0.25,0.25,275,0.0,225,15.966.386.554.621.800,4.495.798.319.327.730,20.588.235.294.117.600,0.0,18.487.394.957.983.100,4.0,238.0000,0.0,0.0000
1,G4,Allegiant Air,MDT,"Harrisburg, PA: Harrisburg International",Harrisburg,PA,Harrisburg International,93.0000,17.0000,0.0000,0.0000,"1,723.0000",3.3000,1.0000,5.5300,0.0000,7.1600,"1,110.0000",58.0000,198.0000,0.0000,357.0000,1.827.956.989.247.310,0.0,0.0,1.827.956.989.247.310,1.013.529.411.764.700,18.526.881.720.430.100,1.941.176.470.588.230,58.823.529.411.764.700,32.529.411.764.705.800,0.0,4.211.764.705.882.350,6.442.251.886.244.920,3.366.221.706.326.170,11.491.584.445.734.100,0.0,20.719.674.985.490.400,16.990.000.000.000.000,"1,723.0000",-999.999.999.999.801,0.0000
2,G4,Allegiant Air,MDW,"Chicago, IL: Chicago Midway International",Chicago,IL,Chicago Midway International,18.0000,3.0000,0.0000,0.0000,58.0000,0.8700,0.7500,1.2500,0.0000,0.1300,20.0000,12.0000,23.0000,0.0000,3.0000,16.666.666.666.666.600,0.0,0.0,16.666.666.666.666.600,19.333.333.333.333.300,32.222.222.222.222.200,0.29,0.25,4.166.666.666.666.660,0.0,43.333.333.333.333.300,3.448.275.862.068.960,20.689.655.172.413.700,39.655.172.413.793.100,0.0,5.172.413.793.103.440,3.0,58.0000,0.0,0.0000
3,G4,Allegiant Air,MEM,"Memphis, TN: Memphis International",Memphis,TN,Memphis International,42.0000,16.0000,0.0000,0.0000,"1,332.0000",3.2000,1.1600,5.5100,0.0000,6.1400,339.0000,290.0000,328.0000,0.0000,375.0000,38.095.238.095.238.000,0.0,0.0,38.095.238.095.238.000,83.25,31.714.285.714.285.700,0.2,725,344.375,0.0,38.375,2.545.045.045.045.040,21.771.771.771.771.700,24.624.624.624.624.600,0.0,28.153.153.153.153.100,16.01,"1,332.0000",10.000.000.000.001.500,0.0000
4,G4,Allegiant Air,MFE,"Mission/McAllen/Edinburg, TX: McAllen Internat...",Mission/McAllen/Edinburg,TX,McAllen International,40.0000,6.0000,0.0000,0.0000,511.0000,2.5900,0.8700,0.6600,0.0000,1.8800,174.0000,71.0000,109.0000,0.0000,157.0000,0.15,0.0,0.0,0.15,8.516.666.666.666.660,12.775,43.166.666.666.666.600,145,0.11,0.0,3.133.333.333.333.330,3.405.088.062.622.300,13.894.324.853.228.900,21.330.724.070.450.000,0.0,30.724.070.450.097.800,6.0,511.0000,0.0,0.0000


## 0.1. Chuẩn hóa biến dùng cho biểu đồ

Mục đích của cell này là đảm bảo notebook vẫn chạy được nếu file CSV thiếu một vài biến phụ. Các tỷ lệ được tính theo dạng **weighted rate** khi tổng hợp, tránh dùng trung bình đơn giản gây lệch do sân bay nhỏ và sân bay lớn có số chuyến rất khác nhau.

In [ ]:
# Chuẩn hóa tên text
for col in ['carrier_name', 'airport', 'airport_name', 'airport_city', 'airport_state', 'airport_full_name']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Nếu thiếu airport_full_name thì dùng airport_name
if 'airport_full_name' not in df.columns:
    df['airport_full_name'] = df['airport_name']

# Đảm bảo numeric columns là số
numeric_cols = [
    'arr_flights', 'arr_del15', 'arr_cancelled', 'arr_diverted', 'arr_delay',
    'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct',
    'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Hàm chuyển các cột tỷ lệ về dạng số.
# Lỗi trước đó xảy ra vì một số cột tỷ lệ được đọc thành string, nên quantile không tính được.
def clean_numeric_series(series):
    """Convert a Series that may contain strings like '19.7%' or '0.197' into numeric values."""
    if series is None:
        return None
    return pd.to_numeric(
        series.astype(str)
              .str.strip()
              .str.replace('%', '', regex=False)
              .str.replace(',', '', regex=False),
        errors='coerce'
    )

safe_flights = df['arr_flights'].replace(0, np.nan)
rate_definitions = {
    'delay_rate': df['arr_del15'] / safe_flights,
    'cancel_rate': df['arr_cancelled'] / safe_flights,
    'diverted_rate': df['arr_diverted'] / safe_flights,
    'problem_rate': (df['arr_del15'] + df['arr_cancelled'] + df['arr_diverted']) / safe_flights
}

for rate_col, calculated_rate in rate_definitions.items():
    if rate_col in df.columns:
        converted = clean_numeric_series(df[rate_col])
        # Nếu cột gốc có giá trị thiếu/không parse được thì thay bằng tỷ lệ tính lại từ count.
        converted = converted.fillna(calculated_rate)
    else:
        converted = calculated_rate

    # Nếu tỷ lệ đang ở dạng 0-100 thay vì 0-1 thì tự nhận diện và đưa về 0-1.
    # Ví dụ 19.7 sẽ thành 0.197, còn 0.197 giữ nguyên.
    valid_values = converted.dropna()
    if len(valid_values) > 0 and valid_values.quantile(0.95) > 1.5:
        converted = converted / 100

    df[rate_col] = converted
    df[rate_col + '_pct'] = df[rate_col] * 100

# Hai biến trung bình cũng cần ép kiểu số nếu file CSV đang lưu dưới dạng string.
if 'avg_delay_min_per_delayed_flight' in df.columns:
    df['avg_delay_min_per_delayed_flight'] = clean_numeric_series(df['avg_delay_min_per_delayed_flight'])
else:
    df['avg_delay_min_per_delayed_flight'] = np.nan

df['avg_delay_min_per_delayed_flight'] = df['avg_delay_min_per_delayed_flight'].fillna(
    df['arr_delay'] / df['arr_del15'].replace(0, np.nan)
)

if 'avg_delay_min_per_arrival' in df.columns:
    df['avg_delay_min_per_arrival'] = clean_numeric_series(df['avg_delay_min_per_arrival'])
else:
    df['avg_delay_min_per_arrival'] = np.nan

df['avg_delay_min_per_arrival'] = df['avg_delay_min_per_arrival'].fillna(
    df['arr_delay'] / safe_flights
)

# Nhãn ngắn để chart không quá dài
carrier_alias = {
    'American Airlines Network': 'American Network',
    'Delta Air Lines Network': 'Delta Network',
    'United Air Lines Network': 'United Network',
    'Southwest Airlines': 'Southwest',
    'SkyWest Airlines Inc.': 'SkyWest',
    'Republic Airline': 'Republic',
    'Endeavor Air Inc.': 'Endeavor',
    'JetBlue Airways': 'JetBlue',
    'Alaska Airlines Network': 'Alaska Network',
    'Hawaiian Airlines': 'Hawaiian',
    'Allegiant Air': 'Allegiant',
    'Spirit Airlines': 'Spirit',
    'Frontier Airlines': 'Frontier',
    'Envoy Air': 'Envoy',
    'PSA Airlines Inc.': 'PSA',
    'Mesa Airlines Inc.': 'Mesa',
    'Horizon Air': 'Horizon',
    'Piedmont Airlines': 'Piedmont',
    'CommuteAir LLC dba CommuteAir': 'CommuteAir'
}
df['carrier_short'] = df['carrier_name'].replace(carrier_alias)
df['airport_label'] = df['airport'].astype(str) + ' — ' + df['airport_full_name'].astype(str)

# Bản dùng cho phân tích tỷ lệ: lọc tối thiểu 20 chuyến để tránh extreme do mẫu quá nhỏ.
# Đây chỉ là lọc phục vụ trực quan hóa, KHÔNG xóa dữ liệu gốc.
analysis_df = df[(df['arr_flights'] >= 20) & (df['delay_rate'].between(0, 1))].copy()

# Nhóm lưu lượng cho box/violin
bins = [20, 50, 200, 1000, np.inf]
labels = ['20–50 flights', '51–200 flights', '201–1,000 flights', '>1,000 flights']
analysis_df['traffic_group'] = pd.cut(analysis_df['arr_flights'], bins=bins, labels=labels, include_lowest=True)

print('Rows in original data:', len(df))
print('Rows used for rate-based charts after minimum-volume filter:', len(analysis_df))
analysis_df[['carrier_name','airport','arr_flights','delay_rate_pct','avg_delay_min_per_delayed_flight']].head()

Rows in original data: 1857
Rows used for rate-based charts after minimum-volume filter: 1642


,carrier_name,airport,arr_flights,delay_rate_pct,avg_delay_min_per_delayed_flight
0,Allegiant Air,MCO,37.0000,10.8108,59.5000
1,Allegiant Air,MDT,93.0000,18.2796,101.3529
3,Allegiant Air,MEM,42.0000,38.0952,83.2500
4,Allegiant Air,MFE,40.0000,15.0000,85.1667
6,Allegiant Air,MLB,25.0000,20.0000,29.2000


## 0.2. Tạo các bảng tổng hợp chính

Các bảng này giúp biểu đồ dùng đúng đơn vị phân tích:

- `carrier_summary`: tổng hợp theo hãng bay.
- `airport_summary`: tổng hợp theo sân bay.
- `cause_summary`: tổng hợp theo nhóm nguyên nhân delay.
- `cause_by_carrier`: cơ cấu nguyên nhân delay theo hãng.
- `priority_df`: điểm ưu tiên hành động theo tổ hợp hãng bay × sân bay.

In [ ]:
def weighted_summary(data, group_cols):
    g = data.groupby(group_cols, as_index=False).agg(
        arr_flights=('arr_flights','sum'),
        arr_del15=('arr_del15','sum'),
        arr_cancelled=('arr_cancelled','sum'),
        arr_diverted=('arr_diverted','sum'),
        arr_delay=('arr_delay','sum')
    )
    g['delay_rate'] = g['arr_del15'] / g['arr_flights'].replace(0, np.nan)
    g['cancel_rate'] = g['arr_cancelled'] / g['arr_flights'].replace(0, np.nan)
    g['diverted_rate'] = g['arr_diverted'] / g['arr_flights'].replace(0, np.nan)
    g['problem_rate'] = (g['arr_del15'] + g['arr_cancelled'] + g['arr_diverted']) / g['arr_flights'].replace(0, np.nan)
    g['avg_delay_min_per_delayed_flight'] = g['arr_delay'] / g['arr_del15'].replace(0, np.nan)
    g['avg_delay_min_per_arrival'] = g['arr_delay'] / g['arr_flights'].replace(0, np.nan)
    for c in ['delay_rate','cancel_rate','diverted_rate','problem_rate']:
        g[c + '_pct'] = g[c] * 100
    return g

carrier_summary = weighted_summary(df, ['carrier_name', 'carrier_short']).sort_values('arr_delay', ascending=False)
airport_summary = weighted_summary(df, ['airport', 'airport_full_name', 'airport_label']).sort_values('arr_delay', ascending=False)

# System KPI: dùng weighted rate toàn hệ thống
system = {
    'total_arr_flights': df['arr_flights'].sum(),
    'total_delayed': df['arr_del15'].sum(),
    'total_cancelled': df['arr_cancelled'].sum(),
    'total_diverted': df['arr_diverted'].sum(),
    'total_delay_minutes': df['arr_delay'].sum()
}
system['delay_rate_pct'] = system['total_delayed'] / system['total_arr_flights'] * 100
system['cancel_rate_pct'] = system['total_cancelled'] / system['total_arr_flights'] * 100
system['diverted_rate_pct'] = system['total_diverted'] / system['total_arr_flights'] * 100
system['problem_rate_pct'] = (system['total_delayed'] + system['total_cancelled'] + system['total_diverted']) / system['total_arr_flights'] * 100
system['avg_delay_min_per_delayed_flight'] = system['total_delay_minutes'] / system['total_delayed']

# Cause mapping
cause_map = {
    'carrier': {'label': 'Carrier-related', 'ct_col': 'carrier_ct', 'delay_col': 'carrier_delay'},
    'weather': {'label': 'Weather', 'ct_col': 'weather_ct', 'delay_col': 'weather_delay'},
    'nas': {'label': 'NAS / airspace system', 'ct_col': 'nas_ct', 'delay_col': 'nas_delay'},
    'security': {'label': 'Security', 'ct_col': 'security_ct', 'delay_col': 'security_delay'},
    'late_aircraft': {'label': 'Late aircraft', 'ct_col': 'late_aircraft_ct', 'delay_col': 'late_aircraft_delay'}
}

cause_rows = []
for cause, meta in cause_map.items():
    cause_rows.append({
        'cause': cause,
        'cause_label': meta['label'],
        'delay_count': df[meta['ct_col']].sum(),
        'delay_minutes': df[meta['delay_col']].sum()
    })
cause_summary = pd.DataFrame(cause_rows)
cause_summary['count_share_pct'] = cause_summary['delay_count'] / cause_summary['delay_count'].sum() * 100
cause_summary['minutes_share_pct'] = cause_summary['delay_minutes'] / cause_summary['delay_minutes'].sum() * 100
cause_summary = cause_summary.sort_values('delay_minutes', ascending=False)

# Cause by carrier for composition charts
cause_by_carrier_rows = []
for _, row in carrier_summary.iterrows():
    carrier_name = row['carrier_name']
    carrier_short = row['carrier_short']
    sub = df[df['carrier_name'] == carrier_name]
    for cause, meta in cause_map.items():
        cause_by_carrier_rows.append({
            'carrier_name': carrier_name,
            'carrier_short': carrier_short,
            'cause': cause,
            'cause_label': meta['label'],
            'delay_minutes': sub[meta['delay_col']].sum()
        })
cause_by_carrier = pd.DataFrame(cause_by_carrier_rows)

# Priority score by carrier-airport combination
priority_df = df[(df['arr_flights'] >= 50) & (df['delay_rate'].between(0, 1))].copy()
priority_df['delay_rate_pct'] = priority_df['delay_rate'] * 100
priority_df['log_arr_flights'] = np.log1p(priority_df['arr_flights'])
priority_df['severity'] = priority_df['avg_delay_min_per_delayed_flight'].replace([np.inf, -np.inf], np.nan)
priority_df = priority_df.dropna(subset=['delay_rate_pct','severity','log_arr_flights'])

# Percentile ranks để 3 yếu tố có cùng thang 0-1
priority_df['rank_delay_rate'] = priority_df['delay_rate_pct'].rank(pct=True)
priority_df['rank_severity'] = priority_df['severity'].rank(pct=True)
priority_df['rank_volume'] = priority_df['log_arr_flights'].rank(pct=True)
priority_df['priority_score'] = (0.4 * priority_df['rank_delay_rate'] +
                                 0.3 * priority_df['rank_severity'] +
                                 0.3 * priority_df['rank_volume']) * 100
priority_df['carrier_airport'] = priority_df['carrier_short'] + ' | ' + priority_df['airport']

print('System KPI:', system)
display(carrier_summary.head())
display(airport_summary.head())
display(cause_summary)

System KPI: {'total_arr_flights': np.float64(660674.0), 'total_delayed': np.float64(130203.0), 'total_cancelled': np.float64(5877.0), 'total_diverted': np.float64(1462.0), 'total_delay_minutes': np.float64(8819601.0), 'delay_rate_pct': np.float64(19.707601631061614), 'cancel_rate_pct': np.float64(0.8895461301640446), 'diverted_rate_pct': np.float64(0.2212891683341557), 'problem_rate_pct': np.float64(20.818436929559812), 'avg_delay_min_per_delayed_flight': np.float64(67.7373102002258)}


,carrier_name,carrier_short,arr_flights,arr_del15,arr_cancelled,arr_diverted,arr_delay,delay_rate,cancel_rate,diverted_rate,problem_rate,avg_delay_min_per_delayed_flight,avg_delay_min_per_arrival,delay_rate_pct,cancel_rate_pct,diverted_rate_pct,problem_rate_pct
2,American Airlines Network,American Network,"84,538.0000","18,592.0000",753.0000,251.0000,"1,419,636.0000",0.2199,0.0089,0.0030,0.2318,76.3574,16.7929,21.9925,0.8907,0.2969,23.1801
15,SkyWest Airlines Inc.,SkyWest,"73,123.0000","14,456.0000","1,336.0000",188.0000,"1,228,119.0000",0.1977,0.0183,0.0026,0.2185,84.9557,16.7952,19.7694,1.8271,0.2571,21.8536
16,Southwest Airlines,Southwest,"117,814.0000","26,258.0000",310.0000,174.0000,"1,214,827.0000",0.2229,0.0026,0.0015,0.2270,46.2650,10.3114,22.2877,0.2631,0.1477,22.6985
4,Delta Air Lines Network,Delta Network,"87,115.0000","12,949.0000",730.0000,165.0000,"820,379.0000",0.1486,0.0084,0.0019,0.1589,63.3546,9.4172,14.8643,0.8380,0.1894,15.8916
18,United Air Lines Network,United Network,"69,451.0000","12,015.0000",375.0000,171.0000,"801,000.0000",0.1730,0.0054,0.0025,0.1809,66.6667,11.5333,17.3000,0.5399,0.2462,18.0861


,airport,airport_full_name,airport_label,arr_flights,arr_del15,arr_cancelled,arr_diverted,arr_delay,delay_rate,cancel_rate,diverted_rate,problem_rate,avg_delay_min_per_delayed_flight,avg_delay_min_per_arrival,delay_rate_pct,cancel_rate_pct,diverted_rate_pct,problem_rate_pct
245,ORD,Chicago O'Hare International,ORD — Chicago O'Hare International,"35,550.0000","9,081.0000","1,336.0000",281.0000,"881,404.0000",0.2554,0.0376,0.0079,0.3009,97.0602,24.7934,25.5443,3.7581,0.7904,30.0928
92,DFW,Dallas/Fort Worth International,DFW — Dallas/Fort Worth International,"25,683.0000","5,228.0000",364.0000,97.0000,"454,707.0000",0.2036,0.0142,0.0038,0.2215,86.9753,17.7046,20.3559,1.4173,0.3777,22.1508
21,ATL,Hartsfield-Jackson Atlanta International,ATL — Hartsfield-Jackson Atlanta International,"29,804.0000","4,233.0000",277.0000,40.0000,"311,744.0000",0.1420,0.0093,0.0013,0.1527,73.6461,10.4598,14.2028,0.9294,0.1342,15.2664
309,SFO,San Francisco International,SFO — San Francisco International,"12,594.0000","5,042.0000",83.0000,40.0000,"297,121.0000",0.4003,0.0066,0.0032,0.4101,58.9292,23.5923,40.0349,0.6590,0.3176,41.0116
91,DEN,Denver International,DEN — Denver International,"25,350.0000","4,059.0000",87.0000,34.0000,"270,645.0000",0.1601,0.0034,0.0013,0.1649,66.6778,10.6763,16.0118,0.3432,0.1341,16.4892


,cause,cause_label,delay_count,delay_minutes,count_share_pct,minutes_share_pct
4,late_aircraft,Late aircraft,"49,767.3100","3,632,094.0000",38.2286,41.1985
0,carrier,Carrier-related,"39,410.7000","2,920,821.0000",30.2732,33.1306
2,nas,NAS / airspace system,"36,946.7100","1,821,766.0000",28.3805,20.6641
1,weather,Weather,"3,848.5700","432,125.0000",2.9563,4.9015
3,security,Security,210.1400,"9,285.0000",0.1614,0.1053


## Chart 01 — KPI Scorecards: Quy mô vấn đề toàn ngành

**Mục trong file chart:** 1. Đánh giá quy mô vấn đề (*Scope of the Problem*)  
**Nhóm mục tiêu trực quan:** KPI baseline / executive summary  
**Câu hỏi trả lời:** Trong tháng 04/2026, bức tranh tổng thể của hệ thống hàng không đang ở mức nào?

**Vì sao dùng KPI cards?**  
Khi cần nêu baseline toàn ngành, người nghe không cần một hình phức tạp. KPI scorecards giúp nhìn ngay tỷ lệ trễ, hủy, chuyển hướng và mức độ trễ trung bình. Đây là phần mở đầu để khán giả hiểu “vấn đề lớn đến mức nào” trước khi đi vào hãng bay, sân bay và nguyên nhân.

In [ ]:
fig = make_subplots(
    rows=1, cols=4,
    specs=[[{'type':'indicator'}, {'type':'indicator'}, {'type':'indicator'}, {'type':'indicator'}]],
    subplot_titles=('Delay rate', 'Cancellation rate', 'Diversion rate', 'Avg minutes per delayed flight')
)

indicators = [
    ('Delay rate', system['delay_rate_pct'], '%'),
    ('Cancellation rate', system['cancel_rate_pct'], '%'),
    ('Diversion rate', system['diverted_rate_pct'], '%'),
    ('Avg delay', system['avg_delay_min_per_delayed_flight'], ' min')
]

for i, (title, value, suffix) in enumerate(indicators, start=1):
    fig.add_trace(go.Indicator(
        mode='number',
        value=value,
        number={'suffix': suffix, 'valueformat': '.2f' if suffix == '%' else '.1f'},
        title={'text': title}
    ), row=1, col=i)

fig.update_layout(
    title='Chart 01 — Scope of the Problem: Baseline toàn ngành trong tháng 04/2026',
    template='plotly_white',
    height=330,
    margin=dict(t=90, b=30),
    annotations=list(fig.layout.annotations) + [
        dict(text='Dữ liệu kỳ báo cáo tháng 04/2026 · Rates được tính theo tổng chuyến bay toàn hệ thống',
             x=0.5, y=-0.10, xref='paper', yref='paper', showarrow=False)
    ]
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_01_kpi_scorecards')

'airline_delay_chart_outputs/chart_01_kpi_scorecards.html'

## Chart 02 — Histogram: Phân phối tỷ lệ delay theo tổ hợp hãng bay × sân bay

**Mục trong file chart:** Bổ sung cho phần *Scope* và *Distribution understanding*  
**Nhóm mục tiêu trực quan:** Distribution  
**Câu hỏi trả lời:** Tỷ lệ delay thường tập trung ở mức nào? Có nhiều tổ hợp có rủi ro delay bất thường không?

**Vì sao dùng Histogram?**  
Histogram dùng cho dữ liệu số liên tục và cho thấy dữ liệu “trải ra” như thế nào. Ở đây, chart chỉ dùng các dòng có `arr_flights >= 20` và `delay_rate` trong khoảng 0–100% để tránh trường hợp sân bay chỉ có 1–2 chuyến làm méo trục. Chart này không dùng để xếp hạng, mà để nhìn hình dạng phân phối và phát hiện đuôi dài/outlier.

In [ ]:
hist_df = analysis_df[analysis_df['delay_rate_pct'].between(0, 100)].copy()

counts, bin_edges = np.histogram(hist_df['delay_rate_pct'], bins=20, range=(0, 100))
total = counts.sum()
percents = counts / total * 100
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_widths = bin_edges[1:] - bin_edges[:-1]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=bin_centers,
    y=percents,
    width=bin_widths * 0.93,
    text=[f"{v:.1f}%" if v >= 0.5 else "" for v in percents],
    textposition="outside",
    textfont=dict(size=10),
    hovertemplate="Delay rate: %{x:.1f}%<br>Share: %{y:.2f}%<extra></extra>",
    marker_color="#636EFA",
    name="Share of records"
))

fig.add_vline(x=system['delay_rate_pct'], line_dash='dash',
              annotation_text='System average', annotation_position='bottom right')

fig.update_layout(
    title='Chart 02 — Distribution: Phân phối tỷ lệ delay theo tổ hợp hãng bay × sân bay',
    xaxis_title='Delay rate (%)',
    yaxis_title='Share of carrier-airport records (%)',
    template='plotly_white',
    height=520,
    bargap=0.05,
    margin=dict(t=80),          # thêm margin top để title không sát số
)
# Tăng range y thêm ~30% so với max để số không chạm title
fig.update_xaxes(range=[0, 100])
fig.update_yaxes(range=[0, max(percents) * 1.30])

display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_02_histogram_delay_rate_distribution')


'airline_delay_chart_outputs/chart_02_histogram_delay_rate_distribution.html'

## Chart 03 — Box Plot: So sánh phân phối tỷ lệ delay giữa các hãng bay lớn

**Mục trong file chart:** 2. Phân tích hiệu suất hãng bay (*Carrier Performance*)  
**Nhóm mục tiêu trực quan:** Distribution + Comparison  
**Câu hỏi trả lời:** Hãng nào có rủi ro delay ổn định hơn, hãng nào có nhiều sân bay bị delay cao bất thường?

**Vì sao dùng Box Plot?**  
Box plot phù hợp khi cần so sánh phân phối của nhiều nhóm cùng lúc. Thay vì chỉ nhìn trung bình delay của một hãng, box plot cho thấy median, độ trải IQR và outlier. Điều này giúp tránh kết luận vội vàng từ một vài sân bay cực đoan.

In [ ]:
## Chart 03 — Dot Plot (Median + IQR range): Carrier delay rate comparison

top_carriers_by_volume = (
    carrier_summary
    .sort_values('arr_flights', ascending=False)
    .head(10)['carrier_name']
)

dot_df = analysis_df[
    analysis_df['carrier_name'].isin(top_carriers_by_volume)
    & analysis_df['delay_rate_pct'].between(0, 100)
].copy()

# Tổng hợp thống kê cho từng hãng
carrier_stats = (
    dot_df.groupby('carrier_short')['delay_rate_pct']
    .agg(
        median='median',
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
        p10=lambda s: s.quantile(0.10),
        p90=lambda s: s.quantile(0.90),
    )
    .reset_index()
    .sort_values('median', ascending=True)  # sắp xếp theo median tăng dần
)

carriers = carrier_stats['carrier_short'].tolist()
sys_avg = system['delay_rate_pct']

fig = go.Figure()

# Dải P10–P90 (mờ hơn, nền)
for _, r in carrier_stats.iterrows():
    fig.add_trace(go.Scatter(
        x=[r['p10'], r['p90']],
        y=[r['carrier_short'], r['carrier_short']],
        mode='lines',
        line=dict(width=6, color='#B0C4DE'),
        opacity=0.5,
        hoverinfo='skip',
        showlegend=False
    ))

# Dải IQR (Q1–Q3, đậm hơn)
for _, r in carrier_stats.iterrows():
    fig.add_trace(go.Scatter(
        x=[r['q1'], r['q3']],
        y=[r['carrier_short'], r['carrier_short']],
        mode='lines',
        line=dict(width=10, color='#4472C4'),
        opacity=0.75,
        hoverinfo='skip',
        showlegend=False
    ))

# Dot median (vòng tròn trắng viền đậm — dễ nhìn trên dải màu)
fig.add_trace(go.Scatter(
    x=carrier_stats['median'],
    y=carrier_stats['carrier_short'],
    mode='markers+text',
    marker=dict(
        symbol='circle',
        size=14,
        color='white',
        line=dict(width=2.5, color='#1F3A6E')
    ),
    text=[f" {v:.1f}%" for v in carrier_stats['median']],
    textposition='middle right',
    textfont=dict(size=10, color='#1F3A6E'),
    name='Median delay rate',
    hovertemplate='<b>%{y}</b><br>Median: %{x:.1f}%<extra></extra>'
))

# Đường baseline toàn ngành
fig.add_vline(
    x=sys_avg,
    line_dash='dash',
    line_color='#E03030',
    line_width=1.5,
    annotation_text=f'System avg: {sys_avg:.1f}%',
    annotation_position='top right',
    annotation_font=dict(size=10, color='#E03030')
)

# Legend giải thích các dải
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
    line=dict(width=6, color='#B0C4DE'), name='P10–P90 range', opacity=0.5))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
    line=dict(width=10, color='#4472C4'), name='IQR (Q1–Q3)', opacity=0.75))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(symbol='circle', size=10, color='white', line=dict(width=2, color='#1F3A6E')),
    name='Median'))

fig.update_layout(
    title='Chart 03 — Dot Plot: Delay rate distribution by carrier (Top 10 by volume)',
    xaxis_title='Delay rate (%)',
    yaxis_title='Carrier',
    template='plotly_white',
    height=580,
    yaxis=dict(categoryorder='array', categoryarray=carriers),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(t=100)
)
fig.update_xaxes(range=[0, min(65, dot_df['delay_rate_pct'].quantile(0.995) * 1.15)])

display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_03_dotplot_carrier_delay_rate_distribution')


'airline_delay_chart_outputs/chart_03_dotplot_carrier_delay_rate_distribution.html'

## Chart 04 — Violin Plot: Độ sâu sự cố theo nhóm lưu lượng sân bay

**Mục trong file chart:** 3. Phân tích điểm nghẽn sân bay (*Airport Bottleneck*)  
**Nhóm mục tiêu trực quan:** Distribution  
**Câu hỏi trả lời:** Những tổ hợp có lưu lượng nhỏ và lớn khác nhau thế nào về số phút trễ trung bình mỗi chuyến bị trễ?

**Vì sao dùng Violin Plot?**  
Violin plot kết hợp density và box plot. Nó không chỉ cho median/IQR mà còn cho biết vùng nào có nhiều quan sát tập trung. Với delay, điều này hữu ích vì “trễ 60 phút” và “trễ 180 phút” tạo ra trải nghiệm rất khác nhau cho hành khách.

In [ ]:
violin_df = analysis_df.dropna(subset=['traffic_group', 'avg_delay_min_per_delayed_flight']).copy()
# Giới hạn trục để chart đọc được, không xóa dữ liệu gốc
upper = violin_df['avg_delay_min_per_delayed_flight'].quantile(0.98)
plot_df = violin_df[violin_df['avg_delay_min_per_delayed_flight'] <= upper]

fig = px.violin(
    plot_df,
    x='traffic_group',
    y='avg_delay_min_per_delayed_flight',
    box=True,
    points='outliers',
    labels={'traffic_group': 'Traffic group', 'avg_delay_min_per_delayed_flight': 'Average delay minutes per delayed flight'},
    title='Chart 04 — Violin Plot: Độ sâu delay theo nhóm lưu lượng chuyến bay'
)
fig.update_layout(template='plotly_white', height=560)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_04_violin_delay_severity_by_traffic_group')

'airline_delay_chart_outputs/chart_04_violin_delay_severity_by_traffic_group.html'

## Chart 05 — Horizontal Bar Chart: Top sân bay tạo tổng phút trễ lớn nhất

**Mục trong file chart:** 3. Phân tích điểm nghẽn sân bay (*Airport Bottleneck*)  
**Nhóm mục tiêu trực quan:** Comparison / Ranking  
**Câu hỏi trả lời:** Nếu nhìn theo thiệt hại thời gian tích lũy, sân bay nào là điểm nghẽn lớn nhất?

**Vì sao dùng Horizontal Bar Chart?**  
Tên sân bay thường dài, nên bar chart nằm ngang đọc tốt hơn column chart. Dữ liệu được sort giảm dần để người xem thấy ngay top bottlenecks. Đây là chart trả lời câu hỏi “cái nào lớn hơn/nhỏ hơn” rất trực tiếp.

In [ ]:
top_airports = airport_summary.head(15).sort_values('arr_delay', ascending=True).copy()

fig = px.bar(
    top_airports,
    x='arr_delay',
    y='airport_label',
    orientation='h',
    text=top_airports['arr_delay'].map(lambda v: f"{v/1e6:.2f}M min"),
    labels={'arr_delay': 'Total arrival delay minutes', 'airport_label': 'Airport'},
    title='Chart 05 — Airport Bottleneck: Top 15 sân bay theo tổng phút trễ'
)
fig.update_traces(
    textposition='inside',
    textfont=dict(size=10, color='white'),
    insidetextanchor='end'
)
# Với cột ngắn (giá trị nhỏ), đặt label ra ngoài để không bị cắt
fig.for_each_trace(lambda t: t.update(
    textposition=[
        'inside' if v > top_airports['arr_delay'].max() * 0.15 else 'outside'
        for v in top_airports['arr_delay']
    ],
    textfont_color=[
        'white' if v > top_airports['arr_delay'].max() * 0.15 else 'black'
        for v in top_airports['arr_delay']
    ]
))
fig.update_layout(template='plotly_white', height=720,
                  xaxis=dict(tickformat=',.0f'))
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_05_bar_top_airports_delay_minutes')


'airline_delay_chart_outputs/chart_05_bar_top_airports_delay_minutes.html'

## Chart 06 — Dot/Lollipop Plot: Tỷ lệ delay của hãng bay so với baseline toàn ngành

**Mục trong file chart:** 2. Phân tích hiệu suất hãng bay (*Carrier Performance*)  
**Nhóm mục tiêu trực quan:** Comparison  
**Câu hỏi trả lời:** Hãng nào có tỷ lệ delay cao hơn mặt bằng chung, hãng nào thấp hơn?

**Vì sao dùng Dot/Lollipop Plot?**  
Dot plot là dạng so sánh tối giản, ít “mực dữ liệu” hơn bar chart nhưng vẫn cho thấy khoảng cách đến baseline. Nó hợp khi muốn người nghe tập trung vào chênh lệch so với mức trung bình toàn hệ thống, không phải vào diện tích cột.

In [ ]:
lollipop_df = carrier_summary[carrier_summary['arr_flights'] >= 500].sort_values('delay_rate_pct', ascending=True).copy()
fig = go.Figure()
base = system['delay_rate_pct']

# Vạch ngang từ baseline đến giá trị hãng
for _, r in lollipop_df.iterrows():
    color = '#EF553B' if r['delay_rate_pct'] > base else '#00CC96'
    fig.add_trace(go.Scatter(
        x=[base, r['delay_rate_pct']],
        y=[r['carrier_short'], r['carrier_short']],
        mode='lines',
        line=dict(width=1.5, color=color),
        showlegend=False,
        hoverinfo='skip'
    ))

# Cục kẹo (marker) — không gắn text vào đây
fig.add_trace(go.Scatter(
    x=lollipop_df['delay_rate_pct'],
    y=lollipop_df['carrier_short'],
    mode='markers',
    marker=dict(size=11, color=[
        '#EF553B' if v > base else '#00CC96'
        for v in lollipop_df['delay_rate_pct']
    ]),
    name='Carrier delay rate',
    hovertemplate='<b>%{y}</b><br>Delay rate: %{x:.2f}%<extra></extra>'
))

# Annotation số — đặt ra ngoài cục kẹo 12px về phía giá trị
# (phải nếu > base, trái nếu < base)
for _, r in lollipop_df.iterrows():
    above = r['delay_rate_pct'] > base
    fig.add_annotation(
        x=r['delay_rate_pct'],
        y=r['carrier_short'],
        text=f"<b>{r['delay_rate_pct']:.1f}%</b>",
        showarrow=False,
        xanchor='left' if above else 'right',
        xshift=8 if above else -8,
        font=dict(size=9.5, color='#333'),
        bgcolor='rgba(255,255,255,0)',
    )

fig.add_vline(x=base, line_dash='dash',
              annotation_text=f'System avg: {base:.1f}%',
              annotation_position='top right',
              line_color='gray')

fig.update_layout(
    title='Chart 06 — Dot/Lollipop: Carrier delay rate vs system baseline',
    xaxis_title='Delay rate (%)',
    yaxis_title='Carrier',
    template='plotly_white',
    height=650,
    showlegend=False,
)
# Mở rộng x-axis để số không bị cắt
x_max = lollipop_df['delay_rate_pct'].max()
fig.update_xaxes(range=[0, x_max * 1.18])
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_06_lollipop_carrier_delay_rate_vs_baseline')


'airline_delay_chart_outputs/chart_06_lollipop_carrier_delay_rate_vs_baseline.html'

## Chart 07 — Bubble Scatter Plot: Ma trận hiệu suất hãng bay

**Mục trong file chart:** 2. Phân tích hiệu suất hãng bay (*Carrier Performance*)  
**Nhóm mục tiêu trực quan:** Relationship + Decision support  
**Câu hỏi trả lời:** Hãng nào vừa có tần suất delay cao, vừa làm hành khách chờ lâu, vừa tạo tổng thiệt hại thời gian lớn?

**Vì sao dùng Bubble Scatter Plot?**  
Scatter plot dùng để xem mối quan hệ giữa hai biến số. Ở đây trục X là tần suất lỗi, trục Y là độ sâu sự cố, kích thước bubble là tổng phút trễ. Góc trên bên phải là vùng cần chú ý nhất vì hội tụ cả rủi ro cao và mức độ trễ nặng.

In [ ]:
bubble_df = carrier_summary.dropna(subset=['delay_rate_pct','avg_delay_min_per_delayed_flight']).copy()

fig = px.scatter(
    bubble_df,
    x='delay_rate_pct',
    y='avg_delay_min_per_delayed_flight',
    size='arr_delay',
    hover_name='carrier_name',
    labels={
        'delay_rate_pct': 'Delay rate (%)',
        'avg_delay_min_per_delayed_flight': 'Avg minutes per delayed flight',
        'arr_delay': 'Total delay minutes'
    },
    title='Chart 07 — Bubble Scatter: Carrier performance matrix'
)
fig.add_vline(x=system['delay_rate_pct'], line_dash='dash',
              annotation_text='System delay rate', annotation_position='top right')
fig.add_hline(y=system['avg_delay_min_per_delayed_flight'], line_dash='dash',
              annotation_text='System avg severity', annotation_position='bottom right')

# Tắt text mặc định, dùng annotation để kiểm soát vị trí từng hãng
fig.update_traces(mode='markers', marker=dict(opacity=0.7, line=dict(width=0.5, color='white')))

# xanchor/yanchor + xshift/yshift để tránh đè nhau
# Tất cả dùng showarrow=False; chỉ shift vài pixel
label_cfg = {
    # carrier_short: (xanchor, yanchor, xshift, yshift)
    'American Network': ('left',   'bottom',  6,  4),
    'SkyWest':          ('right',  'bottom', -6,  4),
    'Southwest':        ('left',   'top',     6, -4),
    'Delta Network':    ('left',   'top',     6, -4),
    'United Network':   ('right',  'top',    -6, -4),   # đè Republic → sang trái
    'PSA':              ('left',   'bottom',  6,  4),
    'Endeavor':         ('left',   'top',     6, -4),
    'Republic':         ('left',   'bottom',  6,  4),   # United đã sang trái
    'Frontier':         ('left',   'bottom',  6,  4),
    'Envoy':            ('right',  'bottom', -6,  4),
    'JetBlue':          ('left',   'top',     6, -4),
    'Spirit':           ('left',   'bottom',  6,  4),
    'GoJet Airlines LLC d/b/a United Express': ('right', 'top', -6, -4),
    'Alaska Network':   ('right',  'top',    -6, -4),
    'Allegiant':        ('right',  'top',    -6, -4),
    'CommuteAir':       ('left',   'bottom',  6,  4),
    'Piedmont':         ('left',   'top',     6, -4),
    'Mesa':             ('right',  'bottom', -6,  4),
    'Horizon':          ('right',  'top',    -6, -4),
}

for _, row in bubble_df.iterrows():
    short = row['carrier_short']
    cfg = label_cfg.get(short, ('left', 'bottom', 6, 4))
    xa, ya, xs, ys = cfg
    fig.add_annotation(
        x=row['delay_rate_pct'],
        y=row['avg_delay_min_per_delayed_flight'],
        text=short,
        showarrow=False,
        xanchor=xa, yanchor=ya,
        xshift=xs, yshift=ys,
        font=dict(size=9, color='#222'),
        bgcolor='rgba(255,255,255,0.75)',
        borderpad=1,
    )

fig.update_layout(template='plotly_white', height=650)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_07_bubble_scatter_carrier_performance')


'airline_delay_chart_outputs/chart_07_bubble_scatter_carrier_performance.html'

## Chart 08 — Scatter Plot: Lưu lượng sân bay và tỷ lệ delay

**Mục trong file chart:** 3. Phân tích điểm nghẽn sân bay (*Airport Bottleneck*)  
**Nhóm mục tiêu trực quan:** Relationship / Outlier detection  
**Câu hỏi trả lời:** Sân bay delay cao vì lưu lượng lớn, hay có những sân bay rủi ro cao bất thường dù lưu lượng không quá lớn?

**Vì sao dùng Scatter Plot?**  
Scatter plot phù hợp khi cần xem quan hệ giữa hai biến số. Trục X dùng `arr_flights` để đo quy mô, trục Y dùng `delay_rate` để đo rủi ro. Các điểm nằm cao bất thường là những sân bay cần kiểm tra kỹ hơn.

In [ ]:
import math

# Chart 08 — bản nhãn sạch, KHÔNG dùng đường chỉ/callout.
# Lưu ý: Chart 08 tổng hợp theo SÂN BAY, không phải hãng bay.
# Mỗi bubble = 1 sân bay; size = tổng phút trễ arrival delay minutes.

air_scatter_df = airport_summary[
    (airport_summary['arr_flights'] >= 50) &
    (airport_summary['delay_rate_pct'].between(0, 100))
].copy()
air_scatter_df = air_scatter_df.dropna(subset=['arr_flights', 'delay_rate_pct', 'arr_delay'])

# Chỉ gắn nhãn top 10 sân bay theo tổng phút trễ.
# Nếu label quá nhiều, cụm sân bay lớn bên phải sẽ rất dễ rối và gây nhầm.
label_airports = air_scatter_df.nlargest(10, 'arr_delay')['airport'].tolist()
bg_df = air_scatter_df[~air_scatter_df['airport'].isin(label_airports)].copy()
hi_df = air_scatter_df[air_scatter_df['airport'].isin(label_airports)].copy()
hi_df = hi_df.sort_values('arr_delay', ascending=False).reset_index(drop=True)

max_delay = max(air_scatter_df['arr_delay'].max(), 1)
sizeref = 2.0 * max_delay / (54 ** 2)

fig = go.Figure()

# Lớp nền: các sân bay còn lại, làm mờ để không tranh thị giác với các bubble chính.
fig.add_trace(go.Scatter(
    x=bg_df['arr_flights'],
    y=bg_df['delay_rate_pct'],
    mode='markers',
    marker=dict(
        size=bg_df['arr_delay'],
        sizemode='area',
        sizeref=sizeref,
        sizemin=4,
        color='#A9B9D6',
        opacity=0.22,
        line=dict(width=0)
    ),
    customdata=bg_df[['airport_label', 'arr_delay']],
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Arrival flights: %{x:,.0f}<br>'
        'Delay rate: %{y:.1f}%<br>'
        'Total delay minutes: %{customdata[1]:,.0f}<extra></extra>'
    ),
    showlegend=False,
    name='Other airports'
))

# Lớp chính: mã sân bay đặt NGAY TRONG bubble.
# Cách này bỏ hoàn toàn đường chỉa/callout, tránh việc người đọc phải đoán label thuộc bubble nào.
fig.add_trace(go.Scatter(
    x=hi_df['arr_flights'],
    y=hi_df['delay_rate_pct'],
    mode='markers+text',
    text=hi_df['airport'],
    textposition='middle center',
    textfont=dict(size=13, color='white'),
    marker=dict(
        size=hi_df['arr_delay'],
        sizemode='area',
        sizeref=sizeref,
        sizemin=18,
        color='#2F66C4',
        opacity=0.92,
        line=dict(width=2.5, color='white')
    ),
    customdata=hi_df[['airport_label', 'arr_delay']],
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Arrival flights: %{x:,.0f}<br>'
        'Delay rate: %{y:.1f}%<br>'
        'Total delay minutes: %{customdata[1]:,.0f}<extra></extra>'
    ),
    showlegend=False,
    name='Top delay airports',
    cliponaxis=False
))

# Baseline toàn hệ thống.
fig.add_hline(
    y=system['delay_rate_pct'],
    line_dash='dash',
    line_color='#E03030',
    line_width=1.4,
    annotation_text=f"System avg: {system['delay_rate_pct']:.1f}%",
    annotation_position='top left',
    annotation_font=dict(size=11, color='#E03030')
)

# Tạo bảng chú giải ở mép phải: không nối bằng đường để tránh rối hình.
airport_short_name = {
    'ORD': "Chicago O'Hare",
    'DFW': 'Dallas/Fort Worth',
    'ATL': 'Atlanta',
    'SFO': 'San Francisco',
    'DEN': 'Denver',
    'LGA': 'LaGuardia',
    'CLT': 'Charlotte',
    'DCA': 'Reagan National',
    'PHX': 'Phoenix',
    'MCO': 'Orlando',
    'LAS': 'Harry Reid',
    'BOS': 'Logan',
    'IAH': 'Houston Intercontinental',
    'LAX': 'Los Angeles',
    'DTW': 'Detroit Metro',
    'PHL': 'Philadelphia',
    'FLL': 'Fort Lauderdale',
    'MIA': 'Miami',
    'MSP': 'Minneapolis–St Paul',
    'JFK': 'John F. Kennedy',
}
legend_lines = [
    f"<b>{row.airport}</b> — {airport_short_name.get(row.airport, str(row.airport_full_name).split('/')[0][:22])} "
    f"({row.delay_rate_pct:.1f}%, {row.arr_delay/1000:.0f}k min)"
    for row in hi_df.itertuples()
]
legend_text = '<b>Labeled airports</b><br>' + '<br>'.join(legend_lines)
fig.add_annotation(
    x=1.02, y=0.98,
    xref='paper', yref='paper',
    text=legend_text,
    showarrow=False,
    align='left',
    xanchor='left', yanchor='top',
    font=dict(size=11, color='#0D2B5E'),
    bgcolor='rgba(255,255,255,0.96)',
    bordercolor='rgba(47,102,196,0.35)',
    borderwidth=1,
    borderpad=8
)

x_min = max(35, air_scatter_df['arr_flights'].min() * 0.70)
x_max = max(110000, air_scatter_df['arr_flights'].max() * 2.30)
y_max = max(53, air_scatter_df['delay_rate_pct'].quantile(0.99) * 1.20)

fig.update_layout(
    title='Chart 08 — Airport Bottleneck: Arrival volume vs delay rate',
    xaxis_title='Arrival flights (log scale)',
    yaxis_title='Delay rate (%)',
    template='plotly_white',
    height=800,
    margin=dict(t=80, r=330, l=85, b=85),
    hoverlabel=dict(bgcolor='white', font_size=12),
    annotations=list(fig.layout.annotations) + [
        dict(
            text='Labels are placed inside the top 10 airports by total delay minutes. Bubble size = total arrival delay minutes.',
            x=0.42, y=-0.13, xref='paper', yref='paper', showarrow=False,
            font=dict(size=11, color='#4A5568')
        )
    ]
)
fig.update_xaxes(
    type='log',
    range=[math.log10(x_min), math.log10(x_max)],
    tickvals=[50, 100, 500, 1000, 5000, 10000, 50000, 100000],
    ticktext=['50', '100', '500', '1k', '5k', '10k', '50k', '100k'],
    showgrid=True,
    gridcolor='rgba(0,0,0,0.08)',
)
fig.update_yaxes(
    range=[0, y_max],
    ticksuffix='%',
    showgrid=True,
    gridcolor='rgba(0,0,0,0.08)',
)
fig.update_traces(cliponaxis=False)

chart_path = save_fig(fig, 'chart_08_scatter_airport_volume_vs_delay_rate_clean_labels')
try:
    import google.colab  # type: ignore
    fig.show(renderer='colab', config={'responsive': True, 'displayModeBar': True})
except Exception:
    display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

print('Saved Chart 08 HTML:', chart_path)


Saved Chart 08 HTML: airline_delay_chart_outputs/chart_08_scatter_airport_volume_vs_delay_rate_clean_labels.html


## Chart 09 — Grouped Bar Chart: So sánh số lượt delay và số phút delay theo nguyên nhân

**Mục trong file chart:** 4. Phân tích cơ cấu nguyên nhân (*Root Cause Breakdown*)  
**Nhóm mục tiêu trực quan:** Comparison + Composition  
**Câu hỏi trả lời:** Nguyên nhân nào xảy ra thường xuyên nhất, và nguyên nhân nào tạo nhiều thời gian chờ nhất?

**Vì sao dùng Grouped Bar Chart?**  
Grouped bar phù hợp khi chỉ có 2–3 sub-groups cần so sánh trực tiếp. Ở đây mỗi nguyên nhân có hai chỉ số: tỷ trọng số lượt delay và tỷ trọng số phút delay. Nếu hai thanh lệch nhau lớn, nguyên nhân đó có “độ sâu” khác với tần suất xảy ra.

In [ ]:
cause_compare = cause_summary[['cause_label','count_share_pct','minutes_share_pct']].melt(
    id_vars='cause_label',
    value_vars=['count_share_pct','minutes_share_pct'],
    var_name='metric',
    value_name='share_pct'
)
cause_compare['metric'] = cause_compare['metric'].replace({
    'count_share_pct': 'Share of delayed incidents',
    'minutes_share_pct': 'Share of delay minutes'
})

fig = px.bar(
    cause_compare,
    x='cause_label',
    y='share_pct',
    color='metric',
    barmode='group',
    text=cause_compare['share_pct'].map(lambda x: f'{x:.1f}%'),
    labels={'cause_label': 'Delay cause', 'share_pct': 'Share (%)', 'metric': 'Metric'},
    title='Chart 09 — Root Cause Breakdown: Frequency share vs minutes share'
)
fig.update_layout(template='plotly_white', height=560)
fig.update_yaxes(range=[0, max(50, cause_compare['share_pct'].max()*1.2)])
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_09_grouped_bar_root_cause_frequency_vs_minutes')

'airline_delay_chart_outputs/chart_09_grouped_bar_root_cause_frequency_vs_minutes.html'

## Chart 10 — 100% Stacked Bar: Cơ cấu nguyên nhân delay theo hãng bay

**Mục trong file chart:** 4. Phân tích cơ cấu nguyên nhân (*Root Cause Breakdown*)  
**Nhóm mục tiêu trực quan:** Composition / Part-to-whole  
**Câu hỏi trả lời:** Với từng hãng bay lớn, tỷ trọng phút trễ đến từ nhóm nguyên nhân nào?

**Vì sao dùng 100% Stacked Bar?**  
Khi muốn so sánh **tỷ trọng** giữa các nhóm có quy mô khác nhau, cần chuẩn hóa về 100%. Chart này không dùng để so sánh tổng phút trễ tuyệt đối, mà dùng để đọc cơ cấu nguyên nhân bên trong từng hãng.

In [ ]:
top_carriers_delay = carrier_summary.head(8)['carrier_name']
stack_df = cause_by_carrier[cause_by_carrier['carrier_name'].isin(top_carriers_delay)].copy()
stack_df['total_carrier_delay'] = stack_df.groupby('carrier_name')['delay_minutes'].transform('sum')
stack_df['share_pct'] = stack_df['delay_minutes'] / stack_df['total_carrier_delay'].replace(0, np.nan) * 100
stack_df['carrier_order'] = stack_df['carrier_name'].map({name: i for i, name in enumerate(top_carriers_delay)})
stack_df = stack_df.sort_values(['carrier_order','cause_label'])

# Chỉ hiện label khi segment đủ rộng (>=5%) để tránh text chồng chéo
stack_df['label_text'] = stack_df['share_pct'].apply(lambda v: f"{v:.1f}%" if v >= 5 else "")

fig = px.bar(
    stack_df,
    x='carrier_short',
    y='share_pct',
    color='cause_label',
    barmode='stack',
    text='label_text',
    labels={'carrier_short': 'Carrier', 'share_pct': 'Share of delay minutes (%)', 'cause_label': 'Cause'},
    title='Chart 10 — 100% Stacked Bar: Cơ cấu phút trễ theo nguyên nhân của top carriers',
    hover_data={'delay_minutes':':,.0f', 'total_carrier_delay':':,.0f', 'share_pct':':.2f'}
)
fig.update_traces(
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(size=10, color='white')
)
fig.update_layout(template='plotly_white', height=620)
fig.update_yaxes(range=[0, 100], ticksuffix='%')
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_10_100_stacked_carrier_cause_composition')


'airline_delay_chart_outputs/chart_10_100_stacked_carrier_cause_composition.html'

## Chart 11 — Treemap: Cơ cấu phân cấp nguyên nhân → hãng bay theo tổng phút trễ

**Mục trong file chart:** 4. Phân tích cơ cấu nguyên nhân (*Root Cause Breakdown*)  
**Nhóm mục tiêu trực quan:** Hierarchical Composition  
**Câu hỏi trả lời:** Trong tổng số phút trễ, nhóm nguyên nhân nào chiếm phần lớn, và trong từng nguyên nhân, hãng nào đóng góp nhiều nhất?

**Vì sao dùng Treemap?**  
Treemap phù hợp với dữ liệu có phân cấp và nhiều thành phần. Ở đây cấp 1 là nhóm nguyên nhân, cấp 2 là hãng bay. Diện tích ô tỷ lệ với tổng phút trễ, giúp người xem vừa thấy bức tranh tổng vừa thấy chi tiết bên trong.

In [ ]:
# Giữ top carriers để treemap không quá rối; phần còn lại gộp thành Others
carrier_total_delay = cause_by_carrier.groupby('carrier_short', as_index=False)['delay_minutes'].sum().sort_values('delay_minutes', ascending=False)
keep_carriers = carrier_total_delay.head(10)['carrier_short'].tolist()
tree_df = cause_by_carrier.copy()
tree_df['carrier_grouped'] = np.where(tree_df['carrier_short'].isin(keep_carriers), tree_df['carrier_short'], 'Other carriers')
tree_df = tree_df.groupby(['cause_label','carrier_grouped'], as_index=False)['delay_minutes'].sum()

# Lọc bỏ ô 0 để treemap không hiện ô trắng
tree_df = tree_df[tree_df['delay_minutes'] > 0].copy()

fig = px.treemap(
    tree_df,
    path=[px.Constant('Total delay minutes'), 'cause_label', 'carrier_grouped'],
    values='delay_minutes',
    title='Chart 11 — Treemap: Delay minutes by cause and carrier',
    color='delay_minutes',
    color_continuous_scale='Blues',
    hover_data={'delay_minutes': ':,.0f'}
)

# texttemplate: hiện tên ô + giá trị triệu phút
# %{label} = tên ô, %{value} = delay_minutes
# Với ô root/cause dùng %{value:,.0f}; với ô carrier dùng format gọn hơn
fig.update_traces(
    texttemplate=(
        "<b>%{label}</b><br>"
        "%{value:,.0f} min"
    ),
    textposition='middle center',
    textfont=dict(size=11),
    insidetextfont=dict(size=11),
    # Ô nhỏ (value nhỏ): Plotly tự ẩn text nếu không đủ chỗ
    # root node ẩn để không bị rối
    root_color='rgba(0,0,0,0)',
)

fig.update_layout(
    template='plotly_white',
    height=680,
    coloraxis_showscale=False,   # ẩn colorbar cho gọn
    margin=dict(t=60, l=10, r=10, b=10)
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_11_treemap_cause_carrier_delay_minutes')


'airline_delay_chart_outputs/chart_11_treemap_cause_carrier_delay_minutes.html'

## Chart 12 — Heatmap Matrix: Điểm ưu tiên hành động theo hãng bay × sân bay

**Mục trong file chart:** 5. Xác định điểm ưu tiên hành động (*Priority Matrix*)  
**Nhóm mục tiêu trực quan:** Decision support / Heatmap Matrix  
**Câu hỏi trả lời:** Tổ hợp hãng bay × sân bay nào cần ưu tiên xử lý trước nếu mục tiêu là giảm rủi ro delay cho hành khách?

**Vì sao dùng Heatmap Matrix?**  
Heatmap hợp khi cần đọc nhanh giá trị trên hai trục phân loại. Ở đây hàng là hãng bay, cột là sân bay, màu là `priority_score`. Score này kết hợp 3 yếu tố: quy mô chuyến bay, tần suất delay và độ sâu sự cố. Ô càng đậm thì càng nên ưu tiên kiểm tra.

In [ ]:
# Chọn các carrier-airport combo có priority cao nhất để matrix không quá rỗng
priority_top = priority_df.sort_values('priority_score', ascending=False).head(80).copy()
selected_carriers = priority_top.groupby('carrier_short')['priority_score'].max().sort_values(ascending=False).head(10).index.tolist()
selected_airports = priority_top.groupby('airport')['priority_score'].max().sort_values(ascending=False).head(14).index.tolist()
heat_df = priority_df[(priority_df['carrier_short'].isin(selected_carriers)) & (priority_df['airport'].isin(selected_airports))].copy()

pivot = heat_df.pivot_table(index='carrier_short', columns='airport', values='priority_score', aggfunc='max')
# Sắp xếp carriers và airports theo điểm max để vùng nóng lên trên/trái
pivot = pivot.loc[pivot.max(axis=1).sort_values(ascending=False).index]
pivot = pivot[pivot.max(axis=0).sort_values(ascending=False).index]

fig = px.imshow(
    pivot,
    text_auto='.0f',
    aspect='auto',
    labels=dict(x='Airport', y='Carrier', color='Priority score'),
    title='Chart 12 — Priority Matrix: Carrier × Airport combinations needing action'
)
fig.update_layout(template='plotly_white', height=650)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
save_fig(fig, 'chart_12_heatmap_priority_matrix')

'airline_delay_chart_outputs/chart_12_heatmap_priority_matrix.html'

## 13. Export minh chứng

Cell cuối lưu các bảng tổng hợp và toàn bộ chart HTML vào file zip. Khi nộp project, nhóm có thể để notebook này trong thư mục `Others`, còn file zip output có thể nộp kèm nếu muốn chứng minh chart được tạo từ code.

In [ ]:
# Lưu các bảng tổng hợp phục vụ Power BI/slide/report nếu cần
summary_dir = 'airline_delay_summary_tables'
os.makedirs(summary_dir, exist_ok=True)

carrier_summary.to_csv(os.path.join(summary_dir, 'carrier_summary.csv'), index=False)
airport_summary.to_csv(os.path.join(summary_dir, 'airport_summary.csv'), index=False)
cause_summary.to_csv(os.path.join(summary_dir, 'cause_summary.csv'), index=False)
cause_by_carrier.to_csv(os.path.join(summary_dir, 'cause_by_carrier.csv'), index=False)
priority_df.to_csv(os.path.join(summary_dir, 'priority_matrix_source.csv'), index=False)

zip_path = 'airline_delay_visualization_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [OUTPUT_DIR, summary_dir]:
        for root, dirs, files in os.walk(folder):
            for file in files:
                full_path = os.path.join(root, file)
                zf.write(full_path, arcname=full_path)

print('Saved chart HTML files:')
for p in saved_charts:
    print('-', p)
print('\nSaved summary tables in:', summary_dir)
print('Zip file:', zip_path)

Saved chart HTML files:
- airline_delay_chart_outputs/chart_01_kpi_scorecards.html
- airline_delay_chart_outputs/chart_02_histogram_delay_rate_distribution.html
- airline_delay_chart_outputs/chart_03_dotplot_carrier_delay_rate_distribution.html
- airline_delay_chart_outputs/chart_04_violin_delay_severity_by_traffic_group.html
- airline_delay_chart_outputs/chart_05_bar_top_airports_delay_minutes.html
- airline_delay_chart_outputs/chart_06_lollipop_carrier_delay_rate_vs_baseline.html
- airline_delay_chart_outputs/chart_07_bubble_scatter_carrier_performance.html
- airline_delay_chart_outputs/chart_08_scatter_airport_volume_vs_delay_rate_clean_labels.html
- airline_delay_chart_outputs/chart_09_grouped_bar_root_cause_frequency_vs_minutes.html
- airline_delay_chart_outputs/chart_10_100_stacked_carrier_cause_composition.html
- airline_delay_chart_outputs/chart_11_treemap_cause_carrier_delay_minutes.html
- airline_delay_chart_outputs/chart_12_heatmap_priority_matrix.html

Saved summary tables 

## Gợi ý đưa vào slide/report

Không nhất thiết đưa cả 12 chart lên slide chính. Nhóm có thể chọn 8–10 chart quan trọng nhất cho slide, còn các chart còn lại để trong appendix hoặc report. Mạch storytelling gợi ý:

1. Mở bằng **Chart 01** để chốt baseline toàn ngành.
2. Dùng **Chart 02–04** để cho thấy delay không phân bố đều và có outlier.
3. Dùng **Chart 05–08** để chỉ ra điểm nóng theo sân bay và hãng bay.
4. Dùng **Chart 09–11** để giải thích delay đến từ nhóm nguyên nhân nào.
5. Kết bằng **Chart 12** để đưa ra danh sách ưu tiên hành động.